# Astrology Notebook (thin UI)

この Notebook は **GitHub 上の `astrology.py` を正本**として読み込み、実行する UI です。  
ロジック本体は `astrology.py` 側に集約し、Notebook は入力設定・実行・表示・ダウンロードに専念します。


In [ ]:
# Cell 1: 初期セットアップ（Colab想定）
import os
from pathlib import Path

REPO_URL = "https://github.com/flareray0/astrology.git"
REPO_DIR = Path("/content/astrology")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("[INFO] repo exists -> git pull")
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}
!pip install -q pyswisseph


In [ ]:
# Cell 2: astrology.py の読み込み（再読み込みしやすく）
import importlib
import astrology

astrology = importlib.reload(astrology)
print("[OK] astrology module loaded:", astrology.__file__)


In [ ]:
# Cell 3: 設定ブロック（ここを編集）
chart_mode = "natal"  # natal / progressed / transit / triple / synastry
PERSON_NAME = "あなた"
PERSON2_NAME = "相手"

# natal
NATAL_DATE = (1984, 11, 15)
NATAL_TIME = "11:27"
NATAL_TZ = 9
NATAL_LAT = 37.38
NATAL_LON = 140.18

# person2 (synastry)
PERSON2_DATE = (1967, 5, 13)
PERSON2_TIME = "00:00"
PERSON2_TZ = 9
PERSON2_LAT = 35.68
PERSON2_LON = 139.65

# progressed
PROGRESS_DATE = (1984, 12, 26)
PROGRESS_TIME = "00:00"
PROGRESS_TZ = 9
PROGRESS_LAT = 37.38
PROGRESS_LON = 140.18

# transit
TRANSIT_DATE = (2026, 2, 12)
TRANSIT_TIME = "00:00"
TRANSIT_TZ = 9
TRANSIT_LAT = 37.38
TRANSIT_LON = 140.18

include_asteroids = True
include_minor_aspects = True
include_composite_aspects = True
hsys = "K"


In [ ]:
# Cell 4: 実行セル（chart_mode 全対応）
if chart_mode not in astrology.CHART_TYPE_LABELS:
    raise ValueError(f"unsupported chart_mode: {chart_mode}")

natal = astrology.build_chart_from_input(NATAL_DATE, NATAL_TIME, NATAL_TZ, NATAL_LAT, NATAL_LON, hsys=hsys, include_asteroids=include_asteroids)
progressed = astrology.build_chart_from_input(PROGRESS_DATE, PROGRESS_TIME, PROGRESS_TZ, PROGRESS_LAT, PROGRESS_LON, hsys=hsys, include_asteroids=include_asteroids)
transit = astrology.build_chart_from_input(TRANSIT_DATE, TRANSIT_TIME, TRANSIT_TZ, TRANSIT_LAT, TRANSIT_LON, hsys=hsys, include_asteroids=include_asteroids)
person2 = astrology.build_chart_from_input(PERSON2_DATE, PERSON2_TIME, PERSON2_TZ, PERSON2_LAT, PERSON2_LON, hsys=hsys, include_asteroids=include_asteroids)

print("=" * 60)
print("チャート種別:", astrology.CHART_TYPE_LABELS[chart_mode])
print("対象日時:", f"{NATAL_DATE} {NATAL_TIME} (UTC+{NATAL_TZ})")
print("対象地点:", f"lat={NATAL_LAT}, lon={NATAL_LON}")
if chart_mode == "synastry":
    print("相手日時:", f"{PERSON2_DATE} {PERSON2_TIME} (UTC+{PERSON2_TZ})")
    print("相手地点:", f"lat={PERSON2_LAT}, lon={PERSON2_LON}")
if chart_mode == "triple":
    print("統合対象: ネイタル + プログレス + トランジット")
print("=" * 60)

if chart_mode == "natal":
    out = astrology.run_natal_report(
        natal,
        person_name=PERSON_NAME,
        include_minor_aspects=include_minor_aspects,
        include_composite_aspects=include_composite_aspects,
    )
elif chart_mode == "progressed":
    out = astrology.run_progressed_report(
        natal,
        progressed,
        person_name=PERSON_NAME,
        include_minor_aspects=include_minor_aspects,
    )
elif chart_mode == "transit":
    out = astrology.run_transit_report(
        natal,
        transit,
        person_name=PERSON_NAME,
        include_minor_aspects=include_minor_aspects,
    )
elif chart_mode == "triple":
    out = astrology.run_triple_report(
        natal,
        progressed,
        transit,
        person_name=PERSON_NAME,
        include_minor_aspects=include_minor_aspects,
    )
else:
    out = astrology.run_synastry_report(
        natal,
        person2,
        person1_name=PERSON_NAME,
        person2_name=PERSON2_NAME,
        include_minor_aspects=include_minor_aspects,
    )

print(out["interpretation"])
print("[SAVE]", out["result_path"], out["interpretation_path"])


In [ ]:
# Cell 5: ダウンロード（Colab）
from google.colab import files

files.download("astrology_result.txt")
files.download("astrology_interpretation.txt")
